# Diabetes Hospital Length of Stay
### Metric: RMSE

**Key findings from EDA:**
- `num_medications × num_lab_procedures` → highest single correlation with target (r=0.508)
- `n_meds_changed` strongly non-linear: 0 changes=4.4d, 2 changes=6.9d
- `readmitted` is available in test and is a strong signal (NO=4.4, >30=5.3, <30=6.1)
- `discharge_disposition_id` encodes clinical complexity (LTC=7.8d vs home=4.0d)
- Numeric columns have jitter + negatives → clip to 0
- A1C × insulin interaction is real and measurable
- Diagnosis combinations (diag_1 + diag_2) carry signal beyond individual codes
- AI Baseline: 2.3128 | Target: beat 2.3177 (current #1)

In [ ]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
pip('lightgbm')
pip('xgboost')
pip('scikit-learn')

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
QUICK_RUN = False   # True = fast test (fewer estimators, 3 folds)

SEED    = 42
N_FOLDS = 3 if QUICK_RUN else 5
TARGET  = 'time_in_hospital'

np.random.seed(SEED)
print(f'QUICK_RUN={QUICK_RUN} | N_FOLDS={N_FOLDS}')

In [ ]:
# Data loading (local portfolio layout)
from pathlib import Path
TRAIN_PATH = Path('train.csv')
TEST_PATH = Path('test.csv')
SUBMIT_PATH = Path('submission.csv')
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
print(f'Train: {train_raw.shape} | Test: {test_raw.shape}')


In [ ]:
# ── RMSE ───────────────────────────────────────────────────────────────────────
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2)))

In [ ]:
# ── Feature Engineering ────────────────────────────────────────────────────────
MED_COLS = [
    'metformin','repaglinide','nateglinide','chlorpropamide','glimepiride',
    'acetohexamide','glipizide','glyburide','tolbutamide','pioglitazone',
    'rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
    'examide','citoglipton','insulin','glyburide-metformin',
    'glipizide-metformin','glimepiride-pioglitazone',
    'metformin-rosiglitazone','metformin-pioglitazone'
]

AGE_MAP = {
    '[0-10)':5,'[10-20)':15,'[20-30)':25,'[30-40)':35,
    '[40-50)':45,'[50-60)':55,'[60-70)':65,'[70-80)':75,
    '[80-90)':85,'[90-100)':95
}

# Clinical severity grouping for discharge disposition
# Higher = more complex / longer expected stay
DISCHARGE_SEVERITY = {
    7:1,   # AMA (Against Medical Advice) - left early
    1:2,   # Home
    2:2, 6:2,        # Home health care
    8:3,             # Home IV therapy
    11:3, 19:3, 20:3, # Expired / other hospital
    13:3, 14:3,      # Hospice
    4:4, 5:4, 9:4,   # ICF / other inpatient
    18:4, 22:4,      # Rehab
    3:4,             # SNF
    15:5, 17:5, 23:5, 27:5,  # LTC / swing bed / long-term
    12:3, 24:2, 25:2, 28:2
}

MED_ENC     = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 3}
A1C_MAP     = {'None': 0, 'Norm': 1, '>7': 2, '>8': 3}
GLU_MAP     = {'None': 0, 'Norm': 1, '>200': 2, '>300': 3}
READMIT_MAP = {'NO': 0, '>30': 1, '<30': 2}


def build_features(df):
    X = df.copy()

    # ── Clip noisy negatives before any computation ──
    for col in ['num_lab_procedures','num_procedures','num_medications',
                'number_outpatient','number_emergency','number_inpatient',
                'number_diagnoses']:
        X[col] = X[col].clip(lower=0)

    # ── Age as numeric ──
    X['age_num'] = X['age'].map(AGE_MAP).fillna(55)

    # ── Medication aggregate features ──
    X['n_meds_active']   = sum((X[c] != 'No').astype(int) for c in MED_COLS)
    X['n_meds_changed']  = sum((X[c].isin(['Up','Down'])).astype(int) for c in MED_COLS)
    X['n_meds_up']       = sum((X[c] == 'Up').astype(int) for c in MED_COLS)
    X['n_meds_down']     = sum((X[c] == 'Down').astype(int) for c in MED_COLS)
    X['n_meds_steady']   = sum((X[c] == 'Steady').astype(int) for c in MED_COLS)
    X['insulin_active']  = (X['insulin'] != 'No').astype(int)
    X['insulin_changed'] = (X['insulin'].isin(['Up','Down'])).astype(int)
    X['insulin_up']      = (X['insulin'] == 'Up').astype(int)

    # ── Encode medication columns numerically ──
    for col in MED_COLS:
        X[col] = X[col].map(MED_ENC).fillna(0)

    # ── Key interactions: treatment intensity ──
    X['med_x_lab']       = X['num_medications'] * X['num_lab_procedures']
    X['med_x_proc']      = X['num_medications'] * X['num_procedures']
    X['lab_x_diag']      = X['num_lab_procedures'] * X['number_diagnoses']
    X['med_x_diag']      = X['num_medications'] * X['number_diagnoses']

    # ── Prior utilization ──
    X['prior_visits']         = X['number_outpatient'] + X['number_emergency'] + X['number_inpatient']
    X['prior_inpatient_flag'] = (X['number_inpatient'] > 0).astype(int)
    X['prior_emergency_flag'] = (X['number_emergency'] > 0).astype(int)
    X['high_utilizer']        = (X['prior_visits'] > 3).astype(int)
    X['inpatient_x_meds']     = X['number_inpatient'] * X['num_medications']

    # ── A1C encoding ──
    X['A1C_num']    = X['A1Cresult'].map(A1C_MAP).fillna(0)
    X['A1C_tested'] = (X['A1Cresult'].notna() & (X['A1Cresult'] != 'None')).astype(int)

    # ── Glucose serum encoding ──
    X['glu_num']    = X['max_glu_serum'].map(GLU_MAP).fillna(0)
    X['glu_tested'] = (X['max_glu_serum'].notna() & (X['max_glu_serum'] != 'None')).astype(int)

    # ── A1C × Insulin interaction ──
    X['a1c_x_insulin_changed'] = X['A1C_num'] * X['insulin_changed']
    X['a1c_x_insulin_up']      = X['A1C_num'] * X['insulin_up']
    X['a1c_x_n_meds_changed']  = X['A1C_num'] * X['n_meds_changed']

    # ── Readmitted encoding ──
    X['readmitted_num'] = X['readmitted'].map(READMIT_MAP).fillna(0)

    # ── Discharge severity ──
    X['discharge_severity'] = X['discharge_disposition_id'].map(DISCHARGE_SEVERITY).fillna(2)

    # ── Admission type ──
    X['is_emergency'] = (X['admission_type_id'] == 1).astype(int)
    X['is_elective']  = (X['admission_type_id'] == 3).astype(int)
    X['is_urgent']    = (X['admission_type_id'] == 2).astype(int)

    # ── Diagnosis combination features ──
    X['diag_12_combo']     = X['diag_1'] + '_' + X['diag_2']
    X['diag_123_combo']    = X['diag_1'] + '_' + X['diag_2'] + '_' + X['diag_3']
    X['all_same_diag']     = ((X['diag_1'] == X['diag_2']) & (X['diag_2'] == X['diag_3'])).astype(int)
    X['has_diabetes_diag'] = ((X['diag_1']=='Diabetes')|(X['diag_2']=='Diabetes')|(X['diag_3']=='Diabetes')).astype(int)
    X['has_circulatory']   = ((X['diag_1']=='Circulatory')|(X['diag_2']=='Circulatory')|(X['diag_3']=='Circulatory')).astype(int)
    X['has_respiratory']   = ((X['diag_1']=='Respiratory')|(X['diag_2']=='Respiratory')|(X['diag_3']=='Respiratory')).astype(int)
    X['has_neoplasm']      = ((X['diag_1']=='Neoplasms')|(X['diag_2']=='Neoplasms')|(X['diag_3']=='Neoplasms')).astype(int)
    X['n_unique_diags']    = X[['diag_1','diag_2','diag_3']].nunique(axis=1)

    # ── Missing indicators ──
    X['race_missing'] = X['race'].isna().astype(int)
    X['race']         = X['race'].fillna('Unknown')

    # ── Binary flags ──
    X['change_bin']      = (X['change'] == 'Ch').astype(int)
    X['diabetesMed_bin'] = (X['diabetesMed'] == 'Yes').astype(int)

    # ── Complexity score ──
    X['complexity'] = (
        X['number_diagnoses'] +
        X['n_meds_active'] / 5 +
        X['num_procedures'] +
        X['discharge_severity']
    )

    # ── Age × diagnosis interaction ──
    X['age_x_diag1_other']  = (X['age_num'] * (X['diag_1'] == 'Other').astype(int))
    X['age_x_neoplasm']     = (X['age_num'] * X['has_neoplasm'])
    X['age_x_n_meds']       = X['age_num'] * X['num_medications']

    return X


train_fe = build_features(train_raw)
test_fe  = build_features(test_raw)
print('Feature engineering done')

In [ ]:
# ── Encode categoricals ────────────────────────────────────────────────────────
CAT_COLS = [
    'race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3',
    'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed', 'readmitted',
    'diag_12_combo', 'diag_123_combo'
]

combined = pd.concat([train_fe, test_fe], axis=0)
for col in CAT_COLS:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))
    train_fe[col] = combined[col].iloc[:len(train_fe)].values
    test_fe[col]  = combined[col].iloc[len(train_fe):].values

DROP_COLS    = ['id', TARGET]
FEATURE_COLS = [c for c in train_fe.columns if c not in DROP_COLS]

X_all  = train_fe[FEATURE_COLS].values.astype(np.float32)
y_all  = train_raw[TARGET].values.astype(np.float32)
X_test = test_fe[FEATURE_COLS].values.astype(np.float32)

print(f'Features: {len(FEATURE_COLS)}')
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1: LightGBM
# ══════════════════════════════════════════════════════════════════════════════
N_EST_LGB = 500 if QUICK_RUN else 3000
LR_LGB    = 0.05 if QUICK_RUN else 0.02

lgb_params = {
    'objective':         'regression_l1',
    'metric':            'rmse',
    'n_estimators':      N_EST_LGB,
    'learning_rate':     LR_LGB,
    'num_leaves':        127,
    'max_depth':         -1,
    'min_child_samples': 20,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'random_state':      SEED,
    'verbose':           -1,
    'n_jobs':            -1,
}

oof_lgb  = np.zeros(len(X_all))
test_lgb = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all)):
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(
        X_all[tr_idx], y_all[tr_idx],
        eval_set=[(X_all[va_idx], y_all[va_idx])],
        callbacks=[
            lgb.early_stopping(150, verbose=False),
            lgb.log_evaluation(False)
        ]
    )
    oof_lgb[va_idx] = model.predict(X_all[va_idx])
    test_lgb += model.predict(X_test) / N_FOLDS
    print(f'  Fold {fold+1}: {rmse(y_all[va_idx], oof_lgb[va_idx]):.4f} | best iter: {model.best_iteration_}')

lgb_rmse = rmse(y_all, oof_lgb)
print(f'\nLGB OOF RMSE: {lgb_rmse:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 2: XGBoost
# ══════════════════════════════════════════════════════════════════════════════
N_EST_XGB = 500 if QUICK_RUN else 3000
LR_XGB    = 0.05 if QUICK_RUN else 0.02

xgb_params = {
    'objective':        'reg:squarederror',
    'eval_metric':      'rmse',
    'n_estimators':     N_EST_XGB,
    'learning_rate':    LR_XGB,
    'max_depth':        6,
    'min_child_weight': 10,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'random_state':     SEED,
    'tree_method':      'hist',
    'device':           'cuda',
    'n_jobs':           -1,
    'verbosity':        0,
}

oof_xgb  = np.zeros(len(X_all))
test_xgb = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_all)):
    model = xgb.XGBRegressor(**xgb_params)
    model.fit(
        X_all[tr_idx], y_all[tr_idx],
        eval_set=[(X_all[va_idx], y_all[va_idx])],
        early_stopping_rounds=150,
        verbose=False
    )
    oof_xgb[va_idx] = model.predict(X_all[va_idx])
    test_xgb += model.predict(X_test) / N_FOLDS
    print(f'  Fold {fold+1}: {rmse(y_all[va_idx], oof_xgb[va_idx]):.4f} | best iter: {model.best_iteration}')

xgb_rmse = rmse(y_all, oof_xgb)
print(f'\nXGB OOF RMSE: {xgb_rmse:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ENSEMBLE: OOF-optimized weights
# ══════════════════════════════════════════════════════════════════════════════
oof_stack  = np.stack([oof_lgb, oof_xgb], axis=1)
test_stack = np.stack([test_lgb, test_xgb], axis=1)

def ensemble_rmse(w):
    w = np.abs(w) / np.abs(w).sum()
    return rmse(y_all, (oof_stack * w).sum(axis=1))

res   = minimize(ensemble_rmse, x0=[0.5, 0.5], method='Nelder-Mead', options={'maxiter': 3000})
opt_w = np.abs(res.x) / np.abs(res.x).sum()

print('=' * 48)
print(f'LGB  OOF RMSE       : {lgb_rmse:.4f}')
print(f'XGB  OOF RMSE       : {xgb_rmse:.4f}')
print(f'Ensemble OOF RMSE   : {ensemble_rmse(opt_w):.4f}')
print(f'Weights — LGB: {opt_w[0]:.3f} | XGB: {opt_w[1]:.3f}')
print('=' * 48)

In [ ]:
# ── Final predictions & submission ────────────────────────────────────────────
final_preds = (test_stack * opt_w).sum(axis=1)
# Clip to valid range [1, 14]
final_preds = np.clip(final_preds, 1, 14)

print(f'Predictions: min={final_preds.min():.2f}, mean={final_preds.mean():.2f}, max={final_preds.max():.2f}')

submission = pd.DataFrame({'id': test_raw['id'], 'prediction': final_preds})
submission.to_csv(SUBMIT_PATH, index=False)
print(f'Saved: {SUBMIT_PATH}')
print(submission.head())